# Train YOLO11n — Fingers
Chạy lần lượt các ô. Bật GPU trong Runtime → Change runtime type. Kết quả được lưu vào Google Drive. Mỗi lần chạy ô train mới sẽ tạo một thư mục riêng.


In [1]:
%pip install -q ultralytics roboflow


In [3]:
from google.colab import drive
import torch
from pathlib import Path

drive.mount("/content/drive")
assert torch.cuda.is_available(), "Hãy bật GPU trong Runtime → Change runtime type."
print(torch.cuda.get_device_name(0))


Mounted at /content/drive
Tesla T4


## Tải dataset
Nhập API key Roboflow khi được hỏi. Key được nhập ẩn. Workspace, project và version lấy từ notebook trước.


In [ ]:
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass("Roboflow API key: "))
project = rf.workspace("hius-workspace-llhmr").project("fingers-ql10t")
dataset = project.version(1).download("yolov11")


In [4]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="4k3QJtWjT33SvxvWbDVs")
project = rf.workspace("hius-workspace-llhmr").project("fingers-ql10t")
version = project.version(1)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Fingers-1 in yolov11:: 100%|██████████| 7204/7204 [00:01<00:00, 5248.27it/s]


In [ ]:
import yaml

root = Path(dataset.location).resolve()
config = yaml.safe_load((root / "data.yaml").read_text())
config["path"] = str(root)
for split, folder in [("train", "train"), ("val", "valid"), ("test", "test")]:
    images = root / folder / "images"
    if images.is_dir():
        config[split] = str(images)
    elif split == "test":
        config.pop("test", None)
    else:
        raise FileNotFoundError(images)
print("Classes:", config["names"])
for split in ("train", "val", "test"):
    if split in config:
        print(split, len(list(Path(config[split]).glob("*"))), "images")
data_yaml = root / "data_colab.yaml"
data_yaml.write_text(yaml.safe_dump(config, allow_unicode=True))


## Train mới
YOLO11n, 100 epochs, 640 px. Checkpoint lưu vào Drive sau mỗi epoch theo mặc định. Giữ runtime kết nối trong khi train.


In [6]:
from ultralytics import YOLO
from datetime import datetime

run_name = "yolo11n_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
model = YOLO("yolo11n.pt")
model.train(
    data="/content/Fingers-1/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    save=True,
    project="/content/drive/MyDrive/Fingers_YOLO/runs",
    name=run_name
)
print("Kết quả:", model.trainer.save_dir)


Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Fingers-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_20261120_012635_735267, nbs=64, nms=None, opset=No

## Xem kết quả
Chạy sau khi train xong.


In [ ]:
from IPython.display import display, Image

run_dir = Path(model.trainer.save_dir)
for name in ["results.png", "confusion_matrix.png"]:
    path = run_dir / name
    if path.exists():
        display(Image(filename=str(path)))
print("Best:", run_dir / "weights" / "best.pt")
print("Last:", run_dir / "weights" / "last.pt")


## Chạy tiếp nếu bị ngắt
Chỉ chạy ô dưới khi lượt train đã dừng. Trong phiên mới, chạy lại các ô cài đặt, mount Drive và tải dataset trước. Dán đúng đường dẫn last.pt của lượt muốn tiếp tục. Không chạy lại ô train mới.


In [ ]:
# Bỏ dấu # ở các dòng dưới để resume lượt train bị ngắt.
# checkpoint = Path(input("Đường dẫn last.pt trên Drive: ").strip())
# assert checkpoint.is_file(), checkpoint
# model = YOLO(str(checkpoint))
# model.train(resume=True, data=str(data_yaml), device=0)
